# Phase 10: Full-Stack Regression & Validation

Full configuration: TS1 gas-phase + CAM Cloud aqueous + TUV-x TS1/TSMLT photolysis.

Checks:
- Bitwise comparison against stored reference output
- Multi-resolution: 480-km and 240-km meshes
- Performance metrics

**Pre-requisite:**
- `data/jw_480km_full/output.nc` (480-km run)
- `data/jw_240km_full/output.nc` (240-km run, optional)
- `data/reference/phase10_reference.nc` (stored reference)

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("..") / "data"
FULL_480 = DATA_DIR / "jw_480km_full" / "output.nc"
FULL_240 = DATA_DIR / "jw_240km_full" / "output.nc"
REFERENCE = DATA_DIR / "reference" / "phase10_reference.nc"

## 1. Regression Test

Bitwise comparison of all chemistry fields against the stored reference.

In [ ]:
if FULL_480.exists() and REFERENCE.exists():
    ds_new = nc.Dataset(FULL_480)
    ds_ref = nc.Dataset(REFERENCE)

    diffs = {}
    for v in ds_ref.variables:
        if v in ds_new.variables and len(ds_ref[v].dimensions) == 3:
            d = np.abs(ds_new[v][:] - ds_ref[v][:]).max()
            if d > 0:
                diffs[v] = d

    if diffs:
        print("DIFFERENCES DETECTED (may be expected after code changes):")
        fig, ax = plt.subplots(figsize=(12, max(4, len(diffs) * 0.3)))
        names = list(diffs.keys())
        vals = [diffs[n] for n in names]
        ax.barh(names, vals, color="tab:orange")
        ax.set_xlabel("Max absolute difference")
        ax.set_title("Regression: Max Differences vs Reference")
        ax.set_xscale("log")
        plt.tight_layout()
        plt.show()
    else:
        print("BITWISE MATCH — no differences detected")

    ds_new.close()
    ds_ref.close()
else:
    print("Need both current run and reference output for regression test")

## 2. Multi-Resolution Summary

Compare key statistics between 480-km and 240-km runs.

In [ ]:
for label, path in [("480-km", FULL_480), ("240-km", FULL_240)]:
    if path.exists():
        ds = nc.Dataset(path)
        nCells = ds.dimensions["nCells"].size
        if "O3" in ds.variables:
            o3 = ds["O3"][-1, :, :]
            print(f"{label} ({nCells} cells): O3 mean={o3.mean():.3e}, "
                  f"max={o3.max():.3e}, min={o3.min():.3e}")
        else:
            print(f"{label} ({nCells} cells): O3 not found")
        ds.close()
    else:
        print(f"{label}: not found")

print("\nPhase 10: Full-stack validation complete")